<a href="https://colab.research.google.com/github/reeyasingh021/Secure-Memo-AI-BreakThroughTech/blob/main/SecureMemo_AI_Coding.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SecureMemo AI Agentic Workflow

In [1]:
# LLM Initialization

In [2]:
import os

from google.colab import userdata

# Configure the API key as an environment variable

os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

print("✓ API key configured successfully!")

✓ API key configured successfully!


### Imports and Installations

In [3]:
# PyPDF and Text Splitting/Chunking Libraries
!pip install -qU langchain-community pypdf langchain-text-splitters
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [4]:
# Google LLM
!pip install -qU langchain-google-genai langchain-community
# Vector Database
!pip install -qU --upgrade chromadb

In [5]:
# Embeddings and Storage
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import Chroma

In [6]:
# Generation
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

## RAG Tool For Project Description

### Company Project Notes - Document Loading and Chunking

In [7]:
# Project Description Notes
loader_pd1 = PyPDFLoader("Project_Descriptions1.pdf")
document_pd1 = loader_pd1.load()

# Combine all pages into one text string
full_text_pd1 = "\n\n".join([page.page_content for page in document_pd1])
print(f"Total text length: {len(full_text_pd1)} characters")
#text = document_pd1[1].page_content

Total text length: 4468 characters


In [8]:
text_splitter_pd1 = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150, separators=["\n\n", "\n", ". ", " ", ""])
chunks_pd1 = text_splitter_pd1.split_text(full_text_pd1)

In [9]:
print(f"Number of chunks: {len(chunks_pd1)}")
print()
for i, chunk in enumerate(chunks_pd1):
    print("Chunk", i, "has",len(chunk), "Characters:")
    print(chunk)
    print()

Number of chunks: 6

Chunk 0 has 950 Characters:
N ‘nd R Associates  
Financial Industry - Ongoing Project Descriptions 
1) Small Business Loan Support Program 
Description: 
 
 The Small Business Loan Support Program helps small and growing businesses secure 
financing to expand operations, purchase equipment, and manage daily expenses. The team 
reviews applications, analyzes financial statements, assesses creditworthiness, and structures 
loan packages, while also tracking repayments and providing guidance on budgeting and cash 
flow management. Using loan management software, spreadsheet analysis tools, and customer 
relationship management systems, employees streamline application processing, maintain 
accurate records, and monitor client interactions. By coordinating these efforts, small and 
medium-sized enterprises will be able to access funding efficiently, reduce the risk of loan 
defaults, and build a foundation for long-term growth. 
Employee Involvement and Access Levels:


### Embeddings and Storage

In [10]:
# Using the model "models/gemini-embedding-001"
# If there is an error with this cell, restart the colab runtime session and run all cells again in order
embeddings_pd1 = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001"
)

# Use Chroma.from_texts() with chunks and embedding model
vectorstore_pd1 = Chroma.from_texts(
    texts=chunks_pd1,
    embedding=embeddings_pd1,
    collection_name="my_documents_collection"
)

print(f"Stored {vectorstore_pd1._collection.count()} chunks in the vector store")

Stored 6 chunks in the vector store


In [11]:
# Preview what is stored in the vector database
sample_pd1 = vectorstore_pd1._collection.peek(limit=1)
print(f"Sample chunk ID: {sample_pd1['ids']}")
print(f"Sample text: {sample_pd1['documents']}")
print(f"Embedding length: {len(sample_pd1['embeddings'][0])}")

Sample chunk ID: ['6b8187c4-f19a-4b81-a620-3785341b8513']
Sample text: ['N ‘nd R Associates  \nFinancial Industry - Ongoing Project Descriptions \n1) Small Business Loan Support Program \nDescription: \n \n The Small Business Loan Support Program helps small and growing businesses secure \nfinancing to expand operations, purchase equipment, and manage daily expenses. The team \nreviews applications, analyzes financial statements, assesses creditworthiness, and structures \nloan packages, while also tracking repayments and providing guidance on budgeting and cash \nflow management. Using loan management software, spreadsheet analysis tools, and customer \nrelationship management systems, employees streamline application processing, maintain \naccurate records, and monitor client interactions. By coordinating these efforts, small and \nmedium-sized enterprises will be able to access funding efficiently, reduce the risk of loan \ndefaults, and build a foundation for long-term growth. \nEm

### Retrieval

In [12]:
query1 = "What are the titles of the ongoing projects in this company?"
results = vectorstore_pd1.similarity_search(query1, k=5)

# Print the results
for i, doc in enumerate(results):
    print(f"Result {i+1}:")
    print(doc.page_content)
    print("---")

Result 1:
N ‘nd R Associates  
Financial Industry - Ongoing Project Descriptions 
1) Small Business Loan Support Program 
Description: 
 
 The Small Business Loan Support Program helps small and growing businesses secure 
financing to expand operations, purchase equipment, and manage daily expenses. The team 
reviews applications, analyzes financial statements, assesses creditworthiness, and structures 
loan packages, while also tracking repayments and providing guidance on budgeting and cash 
flow management. Using loan management software, spreadsheet analysis tools, and customer 
relationship management systems, employees streamline application processing, maintain 
accurate records, and monitor client interactions. By coordinating these efforts, small and 
medium-sized enterprises will be able to access funding efficiently, reduce the risk of loan 
defaults, and build a foundation for long-term growth. 
Employee Involvement and Access Levels:
---
Result 2:
2) Digital Loan Processin

### Generation

In [13]:
llm_pd1 = ChatGoogleGenerativeAI(model="gemini-flash-latest")

template_pd1 = """You are a helpful project description notes agent. Answer the question in a concise and straightforward way. Provide more details about the projects when asked for them. Only use information from this context. If query is out of context, respond that you cannot provide the response."

Context:
{context}

Question: {question}

Answer:"""

prompt_pd1 = ChatPromptTemplate.from_template(template_pd1)

In [14]:
# Helper function to format retrieved documents
def format_docs(docs):
    return "\n\n---\n\n".join(doc.page_content for doc in docs)

# Connect the retriever, prompt, LLM, and output parser
# Use vectorstore.as_retriever(search_kwargs={"k": 3}) for the retriever

rag_chain_pd1 = (
    {"context": vectorstore_pd1.as_retriever(search_kwargs={"k": 3}) | format_docs,
     "question": RunnablePassthrough()}
    | prompt_pd1
    | llm_pd1
    | StrOutputParser()
)

In [15]:
# Test with the provided queries
test_queries_pd1 = [
    "What are the titles of the ongoing projects?",
    "Can you give a summary of each project?",
    "Which project requires the most amount of senior employees with high access levels?",
    "What interns do these projects need?",
    "Can you give me the company's sensitive financial information?"
]

for query in test_queries_pd1:
    print(f"Q: {query}")
    response = rag_chain_pd1.invoke(query)
    print(f"A: {response}")
    print("=" * 60)

Q: What are the titles of the ongoing projects?
A: The titles of the ongoing projects are:
1. Small Business Loan Support Program
2. Digital Loan Processing and Risk Assessment Initiative
3. Personal Investment and Financial Planning Services Program
Q: Can you give a summary of each project?
A: Here is a summary of each project:

1) **Small Business Loan Support Program**: This program assists small businesses in securing financing for expansion and operations. The team manages the full loan lifecycle, including application review, credit assessment, repayment tracking, and budgeting guidance.

2) **Digital Loan Processing and Risk Assessment Initiative**: This initiative focuses on automating loan applications and approvals to increase speed and accuracy. It utilizes risk assessment analytics and workflow tools to evaluate credit risk, detect fraud, and ensure regulatory compliance.

3) **Personal Investment and Financial Planning Services Program**: This program provides customized 

## RAG Tool for Meeting Notes

### Meeting Notes - Document Loading and Chunking

In [16]:
# PyPDF and Text Splitting/Chunking Libraries
# !pip install -qU langchain-community pypdf langchain-text-splitters
# from langchain_community.document_loaders import PyPDFLoader
# from langchain_text_splitters import RecursiveCharacterTextSplitter

In [17]:
# First Meeting Notes to be stored in meeting notes storage
loader_mn1 = PyPDFLoader("Meeting_Notes1.pdf")
document_mn1 = loader_mn1.load()

# Combine all pages into one text string
full_text_mn1 = "\n\n".join([page.page_content for page in document_mn1])
print(f"Total text length: {len(full_text_mn1)} characters")
#text = document_mn1[1].page_content

Total text length: 1582 characters


In [18]:
text_splitter_mn1 = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=50, separators=["\n\n", "\n", ". ", " ", ""])

In [19]:
chunks_mn1 = text_splitter_mn1.split_text(full_text_mn1)

In [20]:
print(f"Number of chunks: {len(chunks_mn1)}")
print()
for i, chunk in enumerate(chunks_mn1):
    print("Chunk", i, "has",len(chunk), "Characters:")
    print(chunk)
    print()

Number of chunks: 11

Chunk 0 has 184 Characters:
Meeting Notes - N 'nd R Associates 
 
● Manager mentioned that maintaining accurate client communications and up-to-date 
reporting is important to ensure overall project coordination.

Chunk 1 has 140 Characters:
● Review loan applications and financial statements to identify areas that may need 
further analysis (Small Business Loan Support Program).

Chunk 2 has 196 Characters:
● Opportunities for process improvements and technology updates were explored, with 
some items flagged for closer review. 
 
● Conduct market research and track investment trends to inform future

Chunk 3 has 191 Characters:
recommendations (Personal Investment and Financial Planning Services Program). 
 
● Updates to client profiles and monitoring progress in portfolio and CRM systems need to 
be done regularly.

Chunk 4 has 123 Characters:
be done regularly. 
 
● Monitor credit risk and repayment trends to catch emerging patterns and review 
potential conce

In [21]:
# Second Meeting Notes
loader_mn2 = PyPDFLoader("Meeting_Notes2.pdf")
document_mn2 = loader_mn2.load()

# Combine all pages into one text string
full_text_mn2 = "\n\n".join([page.page_content for page in document_mn2])
print(f"Total text length: {len(full_text_mn2)} characters")
#text = document_mn2[1].page_content

Total text length: 1423 characters


In [22]:
text_splitter_mn2 = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=50, separators=["\n\n", "\n", ". ", " ", ""])
chunks_mn2 = text_splitter_mn2.split_text(full_text_mn2)

In [23]:
print(f"Number of chunks: {len(chunks_mn2)}")
print()
for i, chunk in enumerate(chunks_mn2):
    print("Chunk", i, "has",len(chunk), "Characters:")
    print(chunk)
    print()

Number of chunks: 9

Chunk 0 has 162 Characters:
Meeting Notes – N 'nd R Associates 
● Review client loan histories and repayment performance using loan management 
software (Small Business Loan Support Program)

Chunk 1 has 134 Characters:
software (Small Business Loan Support Program) 
 
● Validate submitted documents and cross-check compliance requirements with workflow

Chunk 2 has 185 Characters:
automation and document management tools (Digital Loan Processing and Risk 
Assessment Initiative) 
 
● Maintain client data and investment records in CRM and portfolio tracking systems

Chunk 3 has 174 Characters:
● Analyze trends in outstanding loans and assess risk patterns using analytics platforms 
 
● Conduct financial market scans and monitor investment performance with financial

Chunk 4 has 178 Characters:
analytics tools (Personal Investment and Financial Planning Services Program) 
 
● Update and reconcile client portfolios, including investment allocations and projected 
retur

### Embeddings and Storage

In [24]:
# Using the model "models/gemini-embedding-001"
# If there is an error with this cell, restart the colab runtime session and run all cells again in order
embeddings_mn = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001"
)

#meeting_notes_texts = [chunks_mn1, chunks_mn2]

# Use Chroma.from_texts() with chunks and embedding model
vectorstore_mn = Chroma.from_texts(
    texts=chunks_mn1,
    embedding=embeddings_mn,
    collection_name="meeting_notes_collection"
)

vectorstore_mn = Chroma.from_texts(
    texts=chunks_mn2,
    embedding=embeddings_mn,
    collection_name="meeting_notes_collection"
)

print(f"Stored {vectorstore_mn._collection.count()} chunks in the vector store")

Stored 20 chunks in the vector store


In [25]:
# Preview what is stored in the vector database
sample_mn = vectorstore_mn._collection.peek(limit=4)
print(f"Sample chunk ID: {sample_mn['ids']}")
print(f"Sample text: {sample_mn['documents']}")
print(f"Embedding length: {len(sample_mn['embeddings'][0])}")

Sample chunk ID: ['b36e201c-9ff1-4855-8977-01ae0a2b7742', '6990c738-cac4-4751-a074-5fa7e05fda6a', '6a805ba2-a37e-466d-8864-a752400ad383', '30f9ff1e-5fd0-4ed1-b55c-b30149d6d3fb']
Sample text: ["Meeting Notes - N 'nd R Associates \n \n● Manager mentioned that maintaining accurate client communications and up-to-date \nreporting is important to ensure overall project coordination.", '● Review loan applications and financial statements to identify areas that may need \nfurther analysis (Small Business Loan Support Program).', '● Opportunities for process improvements and technology updates were explored, with \nsome items flagged for closer review. \n \n● Conduct market research and track investment trends to inform future', 'recommendations (Personal Investment and Financial Planning Services Program). \n \n● Updates to client profiles and monitoring progress in portfolio and CRM systems need to \nbe done regularly.']
Embedding length: 3072


### Retrieval

In [26]:
# Test retrieval using similarity_search method
query1 = "What tasks are related to the Small Business Loan Support Program?"
results = vectorstore_mn.similarity_search(query1, k=5)

# Print the results
for i, doc in enumerate(results):
    print(f"Result {i+1}:")
    print(doc.page_content)
    print("___")

Result 1:
software (Small Business Loan Support Program) 
 
● Validate submitted documents and cross-check compliance requirements with workflow
___
Result 2:
● Review loan applications and financial statements to identify areas that may need 
further analysis (Small Business Loan Support Program).
___
Result 3:
Meeting Notes – N 'nd R Associates 
● Review client loan histories and repayment performance using loan management 
software (Small Business Loan Support Program)
___
Result 4:
automation and document management tools (Digital Loan Processing and Risk 
Assessment Initiative) 
 
● Maintain client data and investment records in CRM and portfolio tracking systems
___
Result 5:
● Basic updates are needed, such as entering client information, maintaining records, and 
monitoring routine metrics
___


### Generation

In [27]:
# Initialize the LLM (matching the exact format from Project Description)
llm_mn = ChatGoogleGenerativeAI(model="gemini-flash-latest")

template_mn = """You are a helpful assistant analyzing meeting notes for N 'nd R Associates, a financial services company.

Use the following context from meeting notes to answer the question. If you cannot find the answer in the context, say so.

Context:
{context}

Question: {question}

Answer:"""

prompt_mn = ChatPromptTemplate.from_template(template_mn)

# Helper function to format retrieved documents
def format_docs_mn(docs):
    return "\n\n---\n\n".join(doc.page_content for doc in docs)

# Connect the retriever, prompt, LLM, and output parser
rag_chain_mn = (
    {"context": vectorstore_mn.as_retriever(search_kwargs={"k": 3}) | format_docs_mn,
     "question": RunnablePassthrough()}
    | prompt_mn
    | llm_mn
    | StrOutputParser()
)

In [28]:
# Test with the provided queries
test_queries_mn = [
    "What tasks are related to the Small Business Loan Support Program?",
    "What compliance activities were mentioned in the meetings?",
    "What client-related updates need to be done?"
]

for query in test_queries_mn:
    print(f"Q: {query}")
    response = rag_chain_mn.invoke(query)
    print(f"A: {response}")
    print("=" * 60)

Q: What tasks are related to the Small Business Loan Support Program?
A: Based on the meeting notes, the tasks related to the **Small Business Loan Support Program** are:

*   **Document Validation and Compliance:** Validating submitted documents and cross-checking compliance requirements with the workflow using software.
*   **Application Review:** Reviewing loan applications and financial statements to identify areas that may need further analysis.
*   **Performance Tracking:** Reviewing client loan histories and repayment performance using loan management software.
Q: What compliance activities were mentioned in the meetings?
A: According to the meeting notes, the compliance activities mentioned were **verification of client documents and compliance checks**. There was also mention of reviewing pending approvals as a next step.
Q: What client-related updates need to be done?
A: Based on the meeting notes, the following client-related updates need to be done:

*   **Entering client i

## RAG Tool for Employee Data

### Employee Records - Document Loading and Chunking

In [29]:
# Install pandas and openpyxl for Excel file handling
!pip install -qU pandas openpyxl
import pandas as pd

# Load the Excel file
employee_df = pd.read_excel("Sample_Employee_Data.xlsx")

print(f"Loaded {len(employee_df)} employees")
print(f"\nColumns: {list(employee_df.columns)}")
print(f"\nFirst few employees:")
print(employee_df.head())

Loaded 50 employees

Columns: ['Name', 'Email', 'Employee ID', 'Team', 'Position']

First few employees:
              Name                          Email Employee ID         Team  \
0     James Carter     james.carter@securememo.ai      EMP001     Strategy   
1       Priya Nair       priya.nair@securememo.ai      EMP002  Engineering   
2  Marcus Thompson  marcus.thompson@securememo.ai      EMP003    Marketing   
3     Aisha Rahman     aisha.rahman@securememo.ai      EMP004  Engineering   
4    Sofia Delgado    sofia.delgado@securememo.ai      EMP005           HR   

                    Position  
0            Project Manager  
1       System Administrator  
2  Senior Investment Analyst  
3          Data Entry Intern  
4   Client Support Assistant  


In [30]:
# Convert the DataFrame to text format for RAG
# Each employee becomes a document chunk
employee_texts = []
for _, row in employee_df.iterrows():
    employee_text = f"""Employee: {row['Name']}
Email: {row['Email']}
Employee ID: {row['Employee ID']}
Team: {row['Team']}
Position: {row['Position']}"""
    employee_texts.append(employee_text)

print(f"Created {len(employee_texts)} employee documents")
print(f"\nSample employee document:")
print(employee_texts[0])

Created 50 employee documents

Sample employee document:
Employee: James Carter
Email: james.carter@securememo.ai
Employee ID: EMP001
Team: Strategy
Position: Project Manager


In [31]:
# Create additional contextual chunks about teams and positions
# This helps with queries like "who are the engineers?" or "who are the managers?"
team_summary = employee_df.groupby('Team').size().to_dict()
position_summary = employee_df.groupby('Position').size().to_dict()

team_text = "Team Summary:\n" + "\n".join([f"- {team}: {count} employees" for team, count in team_summary.items()])
position_text = "Position Summary:\n" + "\n".join([f"- {position}: {count} employees" for position, count in position_summary.items()])

# Add summary texts to help with aggregate queries
employee_texts.append(team_text)
employee_texts.append(position_text)

print(f"\nTotal chunks including summaries: {len(employee_texts)}")
print(f"\nTeam summary:")
print(team_text)


Total chunks including summaries: 52

Team summary:
Team Summary:
- Engineering: 13 employees
- Finance: 8 employees
- HR: 7 employees
- Marketing: 10 employees
- Strategy: 12 employees


### Embeddings and Storage

In [32]:
# Create embeddings and vector store for employee data
# Using the model "models/gemini-embedding-001"
embeddings_ed = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001"
)

# Create the vector store
vectorstore_ed = Chroma.from_texts(
    texts=employee_texts,
    embedding=embeddings_ed,
    collection_name="employee_data_collection"
)

print(f"Stored {vectorstore_ed._collection.count()} chunks in the employee vector store")

Stored 52 chunks in the employee vector store


In [33]:
# Preview what's stored in the vector database
sample_ed = vectorstore_ed._collection.peek(limit=3)
print(f"Sample employee chunk IDs: {sample_ed['ids'][:3]}")
print(f"\nSample text preview:")
print(sample_ed['documents'][0])
print(f"\nEmbedding length: {len(sample_ed['embeddings'][0])}")

Sample employee chunk IDs: ['261b80f7-0eb0-4909-938a-14910da1ddfc', 'faee9755-9f0e-4e5a-8e5d-1d5b59a9186b', '38a9d948-2a79-4381-957d-ec9d53c35394']

Sample text preview:
Employee: James Carter
Email: james.carter@securememo.ai
Employee ID: EMP001
Team: Strategy
Position: Project Manager

Embedding length: 3072


### Retrieval

In [34]:
# Test retrieval using similarity_search method
query_ed = "Who are the project managers?"
results_ed = vectorstore_ed.similarity_search(query_ed, k=5)

# Print the results
for i, doc in enumerate(results_ed):
    print(f"Result {i+1}:")
    print(doc.page_content)
    print("___")

Result 1:
Employee: Lucas Mendes
Email: lucas.mendes@securememo.ai
Employee ID: EMP036
Team: Strategy
Position: Project Manager
___
Result 2:
Employee: Ethan Scott
Email: ethan.scott@securememo.ai
Employee ID: EMP034
Team: Engineering
Position: Project Manager
___
Result 3:
Position Summary:
- Accounts Officer: 2 employees
- Administrative Intern: 2 employees
- Client Service Executive: 3 employees
- Client Support Assistant: 3 employees
- Compliance Officer: 3 employees
- Credit Analyst: 2 employees
- Data Entry Intern: 2 employees
- Finance Intern: 3 employees
- Finance Manager: 3 employees
- Financial Advisor: 3 employees
- IT Executive: 3 employees
- Junior Business Development Executive: 1 employees
- Loan Administrator: 1 employees
- Loan Support Assistant: 1 employees
- Operations Officer: 2 employees
- Portfolio Manager: 1 employees
- Project Manager: 3 employees
- Risk Analyst: 1 employees
- Senior Credit Risk Analyst: 2 employees
- Senior Investment Analyst: 3 employees
- Sen

### Generation

In [35]:
# Initialize LLM for employee queries
llm_ed = ChatGoogleGenerativeAI(model="gemini-flash-latest")

template_ed = """You are a helpful HR assistant for N 'nd R Associates with access to employee records.

Use the following employee information to answer the question. Be concise and accurate.
If the question asks about multiple people (e.g., "who are the managers?"), list all relevant employees.
If you cannot find the information, say so clearly.

Employee Information:
{context}

Question: {question}

Answer:"""

prompt_ed = ChatPromptTemplate.from_template(template_ed)

# Helper function to format retrieved documents
def format_docs_ed(docs):
    return "\n\n---\n\n".join(doc.page_content for doc in docs)

# Connect the retriever, prompt, LLM, and output parser
# Use vectorstore.as_retriever(search_kwargs={"k": 5}) for the retriever
rag_chain_ed = (
    {"context": vectorstore_ed.as_retriever(search_kwargs={"k": 5}) | format_docs_ed,
     "question": RunnablePassthrough()}
    | prompt_ed
    | llm_ed
    | StrOutputParser()
)

In [36]:
# Test the employee data RAG chain
test_queries_ed = [
    "Who are the project managers?",
    "Which employees work in the Engineering team?",
    "What is Priya Nair's position and email?",
    "Who are the interns?",
    "How many people are on the Finance team?",
    "Who is EMP014?"
]

for query in test_queries_ed:
    print(f"Q: {query}")
    response = rag_chain_ed.invoke(query)
    print(f"A: {response}")
    print("=" * 60)

Q: Who are the project managers?
A: The project managers are Lucas Mendes, Ethan Scott, and James Carter.
Q: Which employees work in the Engineering team?
A: Based on the records provided, the following employees work in the Engineering team:

*   Nadia Okonkwo
*   Ethan Scott
*   Dominic Russo
*   Jonas Berg
Q: What is Priya Nair's position and email?
A: Priya Nair's position is System Administrator and her email is priya.nair@securememo.ai.
Q: Who are the interns?
A: The interns are:
*   **Jasmine Lee** (Data Entry Intern)
*   **Camille Dubois** (Administrative Intern)
*   **Chloe Martin** (Administrative Intern)
*   **Aaliyah Grant** (Finance Intern)
Q: How many people are on the Finance team?
A: There are 8 employees on the Finance team.
Q: Who is EMP014?
A: EMP014 is Tyler Brooks, a Senior Credit Risk Analyst on the Finance team.
